# Keras LSTM baseline

Early univariate baseline with an eight-step input window. Install requirements/keras.txt in a separate environment.

Model logic is preserved from the course project. Paths are relative to this repository; generated files go to `outputs/keras/`. Saved outputs were cleared for publication.


In [ ]:
from pathlib import Path

# Locate the repository when launched from its root or notebooks directory.
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "data").is_dir() and (p / "notebooks").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Launch Jupyter from the repository root.")
DATA_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "keras"
for name in ("train.csv", "val.csv", "test.csv"):
    if not (DATA_DIR / name).is_file():
        raise FileNotFoundError(f"Missing {DATA_DIR / name}. See data/README.md.")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import pandas as pd
import numpy as np

# 读取数据
train_df = pd.read_csv(str(DATA_DIR / 'train.csv'))
val_df = pd.read_csv(str(DATA_DIR / 'val.csv'))
test_df = pd.read_csv(str(DATA_DIR / 'test.csv'))

# 日期解析
def extract_start_date(date_range_str):
    return date_range_str.split(' - ')[0]
def parse_relative_date(relative_str):
    year, month, day = [int(x) for x in relative_str.split('/')]
    return pd.Timestamp(year=year+2000, month=month, day=day)
train_df['start_date'] = train_df['Relative Date'].map(extract_start_date).map(parse_relative_date)
val_df['start_date'] = val_df['Relative Date'].map(extract_start_date).map(parse_relative_date)
test_df['start_date'] = test_df['Relative Date'].map(extract_start_date).map(parse_relative_date)

In [ ]:
# 合并 & 按时间排序
full_train = pd.concat([train_df, val_df], axis=0)
full_train = full_train.sort_values('start_date')
full_series = full_train['Predict Target'].values.astype('float32')

# 归一化 (LSTM一般需要)
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
full_series_scaled = scaler.fit_transform(full_series.reshape(-1,1)).flatten()

In [ ]:
window_size = 8  # 用前8周数据预测第9周

# 训练输入/输出
X = []
y = []
for i in range(len(full_series_scaled) - window_size):
    X.append(full_series_scaled[i:i+window_size])
    y.append(full_series_scaled[i+window_size])
X = np.array(X)
y = np.array(y)

# 最终为 [样本数, window_size, 1]
X = X.reshape(-1, window_size, 1)

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

model = keras.Sequential([
    layers.LSTM(64, input_shape=(window_size, 1)),
    layers.Dense(32, activation='relu'),
    layers.Dense(1)
])
model.compile(optimizer='adam', loss='mse')
# 训练
history = model.fit(X, y, epochs=50, batch_size=8, validation_split=0.1, verbose=2)

In [ ]:
# 取序列最后window_size数作为预测起始
seq = full_series_scaled[-window_size:].tolist()
preds_scaled = []
n_pred = len(test_df)

for i in range(n_pred):
    xx = np.array(seq[-window_size:]).reshape(1, window_size, 1)
    next_pred = model.predict(xx, verbose=0)[0,0]
    preds_scaled.append(next_pred)
    seq.append(next_pred)

# 反归一化
preds = scaler.inverse_transform(np.array(preds_scaled).reshape(-1,1)).flatten()

In [ ]:
# 仅需写 Date 与 Predict Target 列即可
submission = test_df.copy()
submission['Predict Target'] = preds
submission[['Relative Date', 'Predict Target']].to_csv(str(OUTPUT_DIR / 'submission.csv'), index=False)
print('Saved! submission.csv')